In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
#invite people for the Kaggle party
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
# from scipy import stats
import warnings
import time
warnings.filterwarnings('ignore')

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

#bring in the six packs
df_train = pd.read_csv('/home/dias-benchmarks/notebooks/pmarcelino/comprehensive-data-exploration-with-python/input/train.csv')
factor = 3
df_train = pd.concat([df_train]*factor, ignore_index=True)
df_train.info()

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

#check the decoration
df_train.columns

In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

#descriptive statistics summary
df_train['SalePrice'].describe()

In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

#skewness and kurtosis
print("Skewness: %f" % df_train['SalePrice'].skew())
print("Kurtosis: %f" % df_train['SalePrice'].kurt())

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###

#scatter plot grlivarea/saleprice
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

#scatter plot totalbsmtsf/saleprice
var = 'TotalBsmtSF'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%RecordEventWithColumnInfo
### cell 6 ###

#box plot overallqual/saleprice
var = 'OverallQual'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%RecordEventWithColumnInfo
### cell 7 ###

var = 'YearBuilt'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%RecordEventWithColumnInfo
### cell 8 ###

numeric_df = df_train.select_dtypes(include=["number"])
corrmat = numeric_df.corr()

In [ ]:
%%RecordEventWithColumnInfo
### cell 9 ###

#saleprice correlation matrix
k = 10 #number of variables for heatmap
cols = corrmat.sort_values(by='SalePrice', ascending=False).head(k)['SalePrice'].index


cm = np.corrcoef(df_train[cols].values.T)

In [ ]:
%%RecordEventWithColumnInfo
### cell 10 ###

#missing data
total = df_train.isnull().sum().sort_values(ascending=False)
percent = (df_train.isnull().sum()/df_train.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)

In [ ]:
%%RecordEventWithColumnInfo
### cell 11 ###

#dealing with missing data
df_train = df_train.drop(missing_data[missing_data['Total'] > 1].index, axis=1)
df_train.isnull().sum().max() #just checking that there's no missing data missing...

In [ ]:
%%RecordEventWithColumnInfo
### cell 12 ###

#standardizing data
saleprice_scaled = StandardScaler().fit_transform(df_train['SalePrice'].to_numpy()[:, np.newaxis])
low_range = saleprice_scaled[saleprice_scaled[:,0].argsort()][:10]
high_range= saleprice_scaled[saleprice_scaled[:,0].argsort()][-10:]
print(low_range)
print(high_range)

In [ ]:
%%RecordEventWithColumnInfo
### cell 13 ###

#bivariate analysis saleprice/grlivarea
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%RecordEventWithColumnInfo
### cell 14 ###

#deleting points
df_train.sort_values(by = 'GrLivArea', ascending = False)[:2]
df_train = df_train.drop(df_train[df_train['Id'] == 1299].index)
df_train = df_train.drop(df_train[df_train['Id'] == 524].index)

In [ ]:
%%RecordEventWithColumnInfo
### cell 15 ###

#bivariate analysis saleprice/grlivarea
var = 'TotalBsmtSF'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%RecordEventWithColumnInfo
### cell 16 ###

#applying log transformation
df_train['SalePrice'] = np.log(df_train['SalePrice'])

In [ ]:
%%RecordEventWithColumnInfo
### cell 17 ###

#data transformation
df_train['GrLivArea'] = np.log(df_train['GrLivArea'])

In [ ]:
%%RecordEventWithColumnInfo
### cell 18 ###

#create column for new variable (one is enough because it's a binary categorical feature)
#if area>0 it gets 1, for area==0 it gets 0
df_train['HasBsmt'] = pd.Series(len(df_train['TotalBsmtSF']), index=df_train.index)
df_train['HasBsmt'] = 0 
df_train.loc[df_train['TotalBsmtSF']>0,'HasBsmt'] = 1

In [ ]:
%%RecordEventWithColumnInfo
### cell 19 ###

#transform data
df_train.loc[df_train['HasBsmt']==1,'TotalBsmtSF'] = np.log(df_train['TotalBsmtSF'])

In [ ]:
%%RecordEventWithColumnInfo
### cell 20 ###

#histogram and normal probability plot
_ = df_train[df_train['TotalBsmtSF']>0]['TotalBsmtSF']
_ = df_train[df_train['TotalBsmtSF']>0]['TotalBsmtSF']

In [ ]:
%%RecordEventWithColumnInfo
### cell 21 ###

#scatter plot
_ = df_train[df_train['TotalBsmtSF']>0]['TotalBsmtSF']
_ = df_train[df_train['TotalBsmtSF']>0]['SalePrice']

In [ ]:
%%RecordEventWithColumnInfo
### cell 22 ###

#convert categorical variable into dummy
df_train = pd.get_dummies(df_train)